In [1]:
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import MultiLabelBinarizer

from pathlib import Path

In [2]:
DAT_DIR=next((d for d in [Path("data/procesed"), Path("../data/processed"),Path(".")]
              if (d/ "parsed_netflix_data.parquet").exists()), Path("data/procesed"))
INPUT_PATH=DAT_DIR / "parsed_netflix_data.parquet"
OUTPUT_PATH=DAT_DIR / "parsed_netflix_data_features.parquet"
SEED=42


In [3]:
CONTRACTIONS={
    
}

In [6]:
STOPWORDS=frozenset(ENGLISH_STOP_WORDS).union(CONTRACTIONS)
MIN_TOKEN_LEN=2
GENRE_PREFIX="genre_"
EXCLUDE_GENRES= set()

COUNTRY_SKIP_TOKENS={"unknown","other"}
DIRECTOR_WEIGHT=1

LIST_COLS=["genres","directors","cast_list","countries"]
REQUIRED_COLS=["description","genres","directors","cast_list","primary_country","release_year","year_added","type","duration_min","seasons"]

In [ ]:
def to_list(x):
    if x is None:
        return []
    if isinstance(x,(list,tuple,np.ndarray)):
        return [str(v) for v in x if v is not None and str(v).strip()!=""]
    if isinstance(x,float) and np.isnan(x):
        return []
    return [str(x).strip()]


def load_input(df):
    df=df.copy()
    if "duration" in df.columns:
        df=df.drop(columns=["duration"])
    if "duration_mins" in df.columns and  "duration_min" not in df.columns:
        df=df.rename(columns={"duration_mins":"duration_min"})

    missing_cols=[c for c in REQUIRED_COLS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    for col in LIST_COLS:
        df[col]=df[col].apply(to_list)


    for c in ["release_year","year_added","month_added"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c],errors="coerce").astype("Int64")

    df["description"]=df["description"].fillna("").astype(str)
    return df
    
              

In [31]:
raw_df=pd.read_parquet(INPUT_PATH)
print("shape:",raw_df.shape)
print("columns:",raw_df.columns.tolist())

shape: (8803, 22)
columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'maturity_rating', 'duration', 'listed_in', 'description', 'year_added', 'month_added', 'duration_mins', 'seasons', 'maturity_group', 'genres', 'directors', 'cast_list', 'countries', 'primary_country']


In [32]:
df=load_input(raw_df)
print("Afterr loading input, shape:",df.shape)

Afterr loading input, shape: (8803, 22)


In [33]:
assert isinstance(df["genres"].iloc[0], list)
assert "duration" not in df.columns and "duration_min" in df.columns

In [41]:
def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if unicodedata.category(c) != 'Mn')
def clean_words(words):
    cleaned=[]
    for w in words:
        w=w.lower().strip()
        w=strip_accents(w)
        if len(w)<MIN_TOKEN_LEN or w in STOPWORDS:
            continue
        cleaned.append(w)
    return cleaned

def clean_text(desc):
    if not isinstance(desc,str):
        return ""
    desc=desc.lower()
    desc=strip_accents(desc)
    desc=re.sub(r"[^a-z0-9\s]"," ",desc)
    desc=re.sub(r"\s+"," ",desc)
    words=desc.split()
    words=clean_words(words)
    return " ".join(words)


def clean_description(df):
    df=df.copy()
    df["description_clean"]=df["description"].apply(clean_text)
    return df




In [42]:
df1=clean_description(df)
assert not df1["description_clean"].str.contains(r"[^a-z0-9\s]").any()
assert not df1["description_clean"].str.contains(r"\bdont\b").any()

In [47]:
def clean_names(s):
    s=strip_accents(s).lower().replace("&","and")
    return re.sub(r"[^a-z0-9]+","_",s).strip("_")

def add_genres(df):
    df=df.copy()
    genres= df["genres"].apply(lambda gs:[g for g in gs if g not in EXCLUDE_GENRES])
    mlb=MultiLabelBinarizer()
    onehot =pd.DataFrame(mlb.fit_transform(genres),index=df.index,
                         columns=[GENRE_PREFIX + clean_names(g) for g in mlb.classes_]).astype("int8")

    return pd.concat([df,onehot],axis=1)

def genre_cols(df):
    return [c for c in df.columns if c.startswith(GENRE_PREFIX)]

In [53]:
df2=add_genres(df1)
expected=df2["genres"].apply(lambda gs:len([g for g in gs if g not in EXCLUDE_GENRES]))
assert (df2[genre_cols(df2)].sum(axis=1) == expected).all()
print(len(genre_cols(df2)), "genre columns")
for lab in ["Movies","TV Shows"]:
    print(f"titles whose only genre is {lab} : ", (df["genres"].apply(lambda g:g ==[lab])).sum())
df['genres'].explode().value_counts()

42 genre columns
titles whose only genre is Movies :  57
titles whose only genre is TV Shows :  16


genres
International Movies            2751
Dramas                          2424
Comedies                        1673
International TV Shows          1350
Documentaries                    869
Action & Adventure               859
TV Dramas                        762
Independent Movies               755
Children & Family Movies         641
Romantic Movies                  616
TV Comedies                      581
Thrillers                        575
Crime TV Shows                   470
Kids' TV                         451
Docuseries                       395
Music & Musicals                 375
Romantic TV Shows                370
Horror Movies                    357
Stand-Up Comedy                  343
Reality TV                       255
British TV Shows                 253
Sci-Fi & Fantasy                 243
Sports Movies                    219
Anime Series                     176
Spanish-Language TV Shows        174
TV Action & Adventure            168
Korean TV Shows                

In [68]:
def add_derived(df):
    df=df.copy()
    df["content_age"]=(df["year_added"]-df["release_year"]).clip(lower=0)
    df["is_movie"]=(df["type"]=="Movie").astype("int8")
    df["desc_len"]=df["description"].str.split().str.len()
    df["cast_count"]=df["cast_list"].apply(len)
    df["director_count"]=df["directors"].apply(len)
    return df

In [69]:
raw_age=df2["year_added"]-df2["release_year"]
print("negative", (raw_age<=0).sum())
print(df2.loc[(raw_age<0).fillna(False),"type"].value_counts())
df3=add_derived(df2)

negative 3253
type
TV Show    12
Movie       2
Name: count, dtype: int64


In [70]:
def name_token(name):
    return re.sub(r"[\W_]+","" ,strip_accents(name).lower())

def make_soup(row):
    parts=[row["description_clean"]]
    parts+= [name_token(n) for n in row["directors"]]
    parts+= [name_token(n) for n in row["cast_list"]]
    country=name_token(row["primary_country"])
    if country not in COUNTRY_SKIP_TOKENS:
        parts.append(country)

    return " ".join(p for p in parts if p)

def add_soup(df):
    df=df.copy()
    df["soup"]=df.apply(make_soup,axis=1)
    return df


        

In [71]:
df4=add_soup(df3)
genre_tokens={name_token(g) for g in df["genres"].explode().dropna().unique() if " " in g}


for i in df4.sample(3,random_state=SEED).index:
    print(df4.at[i,"title"],"-> ",df4.at[i,"soup"], "\n")

The Circle ->  status strategy collide social experiment competition online players flirt befriend catfish way 100 000 michellebuteau unitedstates 

The Old Thieves: The Legend of Artegios ->  carrying prison sentences notorious thieves mexico city share principles preferred methods respect craft everardogonzalez mexico 

One Last Shot ->  low budget short film best buddies meet night booze fueled fun moves vancouver mikeclattenburg robbwells johnpaultremblay johndunsworth canada 



In [74]:
candidates=["is_movie","duration_min", "seasons","release_year","year_added","month_added","content_age","maturity_group","primary_country","desc_len","cast_count","director_count"]

profile=pd.DataFrame({
    "missing": df4[candidates].isna().mean().mul(100).round(1),
    "top_value_share":[df4[c].value_counts(normalize=True,dropna=False).iloc[0].round(3) for c in candidates]
})

profile["near_constant"]=profile["top_value_share"]>0.95
profile

,missing,top_value_share,near_constant
is_movie,0.0,0.696,False
duration_min,30.4,0.304,False
seasons,69.6,0.696,False
release_year,0.0,0.130,False
year_added,0.1,0.229,False
month_added,0.1,0.094,False
content_age,0.1,0.370,False
maturity_group,0.0,0.455,False
primary_country,0.0,0.365,False
desc_len,0.0,0.151,False


In [76]:
numeric=df4[candidates].select_dtypes("number").astype(float)
corr= numeric.corr(method="spearman")
corr.round(2)

,is_movie,duration_min,seasons,release_year,year_added,month_added,content_age,desc_len,cast_count,director_count
is_movie,1.00,NaN,NaN,-0.23,-0.03,-0.02,0.23,0.08,0.06,0.82
duration_min,NaN,1.00,NaN,-0.19,0.13,0.01,0.26,0.08,0.43,-0.01
seasons,NaN,NaN,1.00,0.02,0.10,0.00,-0.00,0.01,0.15,-0.08
release_year,-0.23,-0.19,0.02,1.00,0.37,-0.04,-0.85,0.00,-0.14,-0.19
year_added,-0.03,0.13,0.10,0.37,1.00,-0.16,0.05,0.09,0.10,-0.02
month_added,-0.02,0.01,0.00,-0.04,-0.16,1.00,-0.08,0.01,0.01,-0.02
content_age,0.23,0.26,-0.00,-0.85,0.05,-0.08,1.00,0.03,0.21,0.19
desc_len,0.08,0.08,0.01,0.00,0.09,0.01,0.03,1.00,0.24,0.06
cast_count,0.06,0.43,0.15,-0.14,0.10,0.01,0.21,0.24,1.00,0.04
director_count,0.82,-0.01,-0.08,-0.19,-0.02,-0.02,0.19,0.06,0.04,1.00


In [77]:
assert len(df4)==len(df)

In [80]:
OUTPUT_PATH = Path("../data/processed/features_netflix_data.parquet")

df4.to_parquet(OUTPUT_PATH,index=False)
print("saved to", OUTPUT_PATH)

saved to ../data/processed/features_netflix_data.parquet
